### Wave F — Agent performance

Agent = `assignedto` → `salesexecutives.user` → `users.email`.

| Sheet | Yellow box |
|-------|------------|
| `Agent_Efficiency` | #20, #21, #23 |
| `Agent_Response_Resolution` | #22 |



In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import START_TS, END_TS, CLOSURE_STAGE, q, write_excel

leads = q(f"""
SELECT CAST(l._id AS varchar) lead_oid, l.createdat born_at,
       COALESCE(NULLIF(TRIM(CAST(u.email AS varchar)), ''), CAST(se._id AS varchar)) agent_email
FROM leads l
LEFT JOIN salesexecutives se ON CAST(se._id AS varchar)=CAST(l.assignedto AS varchar)
LEFT JOIN users u ON CAST(u._id AS varchar)=CAST(se.user AS varchar)
WHERE l.createdat >= {START_TS} AND l.createdat < {END_TS}
""")
closed = set(q(f"SELECT CAST(lead AS varchar) lead_oid FROM leadstagehistories WHERE CAST(stage AS varchar)='{CLOSURE_STAGE}' AND createdat >= {START_TS} AND createdat < {END_TS} GROUP BY 1")["lead_oid"])
psv = set(q(f"SELECT CAST(lead AS varchar) lead_oid FROM leadstagehistories WHERE CAST(stage AS varchar)='Post Site Visit' AND createdat >= {START_TS} AND createdat < {END_TS} GROUP BY 1")["lead_oid"])
calls = q(f"""
SELECT CAST(lead AS varchar) lead_oid, COUNT(*) call_count, AVG(TRY_CAST(callduration AS double)) mean_duration_sec
FROM leadcallrecords WHERE createdat >= {START_TS} AND createdat < {END_TS} GROUP BY 1
""")
first_call = q(f"""
SELECT CAST(lead AS varchar) lead_oid, MIN(createdat) first_call_at
FROM leadcallrecords WHERE createdat >= {START_TS} AND createdat < {END_TS} GROUP BY 1
""")
leads["is_closed"] = leads["lead_oid"].isin(closed).astype(int)
leads["is_psv"] = leads["lead_oid"].isin(psv).astype(int)
base = leads.merge(calls, on="lead_oid", how="left")
base["call_count"] = base["call_count"].fillna(0)
eff = base.groupby("agent_email", as_index=False).agg(**{
    "Lead count": ("lead_oid", "nunique"),
    "PSV count": ("is_psv", "sum"),
    "Sales closure count": ("is_closed", "sum"),
    "Total calls": ("call_count", "sum"),
    "Mean call duration sec": ("mean_duration_sec", "mean"),
}).sort_values("Sales closure count", ascending=False)
eff["Mean call duration sec"] = eff["Mean call duration sec"].round(1)
resp = leads.merge(first_call, on="lead_oid", how="inner")
resp["response_hours"] = (pd.to_datetime(resp["first_call_at"]) - pd.to_datetime(resp["born_at"])).dt.total_seconds() / 3600.0
f_resp = resp.groupby("agent_email", as_index=False).agg(**{"n":("lead_oid","nunique"),"Median response hours":("response_hours","median"),"Mean response hours":("response_hours","mean")})
f_resp["Median response hours"] = f_resp["Median response hours"].round(2)
f_resp["Mean response hours"] = f_resp["Mean response hours"].round(2)
write_excel(ROOT / "Agent_Performance_Analysis.xlsx", {
    "Agent_Efficiency": eff,
    "Agent_Response_Resolution": f_resp.sort_values("Median response hours"),
})
print("agents", eff["agent_email"].nunique())

